In [ ]:
# Préparation commune des TP Python
# Le notebook utilise uniquement les ressources fournies dans le dossier codes/.
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des paquets manquants :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

WORKDIR = Path.cwd().resolve()
CODES_DIR = WORKDIR.parent if WORKDIR.name == "correction" else WORKDIR
TOOLBOX_DIR = CODES_DIR / "toolbox"
DATA_DIR = CODES_DIR / "data"
if not CODES_DIR.is_dir():
    raise FileNotFoundError("Exécutez le notebook depuis le dossier codes/." )
if TOOLBOX_DIR.is_dir():
    sys.path.insert(0, str(TOOLBOX_DIR))

DICE_FILE = TOOLBOX_DIR / "DICE.py"
if not DICE_FILE.is_file():
    raise FileNotFoundError(
        f"Module du cours introuvable : {DICE_FILE}. "
        "Téléchargez le dossier codes complet, avec toolbox/."
    )

DATA_FILE = DATA_DIR / "SSP_scenarios.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Donnée du cours introuvable : {DATA_FILE}. "
        "Téléchargez le dossier codes complet, avec data/."
    )

print("Environnement prêt :", CODES_DIR)


# TP3 — Génération de scénarios physiques SSP

**Date de la séance :** mercredi 9 septembre 2026, 08:30–10:30

**Objectifs**
- Manipuler les scénarios SSP (Shared Socioeconomic Pathways)
- Simuler plusieurs scénarios physiques avec DICE
- Calculer des probabilités de dépassement de seuils de température
- Identifier les années de dépassement de +1,5°C et +2°C selon le scénario

**Prérequis** : TP2 (prise en main de DICE).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import DICE

## Partie 1 — Scénarios SSP

**Question 1.** Chargez le fichier SSP_scenarios.csv. Affichez les premières lignes et les colonnes disponibles.

In [ ]:
# Expected output: colonnes ['year', 'scenario', 'emissions_gtco2'], 5 scénarios SSP1-SSP5
df_ssp = pd.read_csv(DATA_DIR / "SSP_scenarios.csv")
print(df_ssp.columns.tolist())
print(df_ssp['scenario'].unique())
df_ssp.head()

## Partie 2 — Simulation des scénarios SSP avec DICE

**Question 2.** Simulez trois scénarios avec DICE en faisant varier les paramètres :
- SSP1 (soutenabilité) : `gA=0.04`, `Linf=8500`, `gsig=0.025`
- SSP2 (référence) : paramètres par défaut
- SSP5 (fossile) : `gA=0.10`, `Linf=12000`, `gsig=0.008`

In [ ]:
ssp_params = {
    'SSP1': dict(gA=0.04, Linf=8500, gsig=0.025),
    'SSP2': dict(),  # default
    'SSP5': dict(gA=0.10, Linf=12000, gsig=0.008)
}
ssp_paths = {}
for name, kwargs in ssp_params.items():
    p = DICE.Params(**kwargs)
    path = DICE.init_states(p)
    timevec = range(1, p.nT)
    path[1:, p.i_mu] = 0.03
    path = DICE.update_path(path, timevec, p)
    ssp_paths[name] = (p, path)

In [ ]:
# Expected output: SSP5 réchauffe le plus vite, SSP1 le moins vite
plt.figure(figsize=(9, 5))
for name, (p, path) in ssp_paths.items():
    plt.plot(path[:, p.i_time], path[:, p.i_T_AT], label=name, linewidth=2)
plt.xlabel('Année'); plt.ylabel('T_AT (°C)')
plt.title('Trajectoires de température par scénario SSP')
plt.legend(); plt.grid(True)
plt.show()

**Question 3.** Calculez la probabilité de dépasser +1,5°C, +2°C et +3°C en 2100 pour chaque scénario (à ce stade, avec 3 scénarios déterministes, la 'probabilité' est 0 ou 1 — discutez pourquoi une vraie distribution nécessite Monte Carlo).

In [ ]:
# Expected output: tableau binaire 0/1 par scénario et seuil
thresholds = [1.5, 2.0, 3.0]
rows = []
for name, (p, path) in ssp_paths.items():
    T_2100 = path[-1, p.i_T_AT]
    row = {'scenario': name, 'T_AT_2100': T_2100}
    for th in thresholds:
        row[f'P(T>{th})'] = int(T_2100 > th)
    rows.append(row)
df_proba = pd.DataFrame(rows)
df_proba

## Partie 3 — Analyse des risques chroniques et aigus

**Question 4.** Pour chaque scénario, calculez la première année de dépassement de +1,5°C et de +2°C. Présentez les résultats dans un tableau.

In [ ]:
# Expected output: SSP5 dépasse les seuils plus tôt que SSP1 ; certains scénarios ne dépassent jamais +2°C sur l'horizon simulé
rows = []
for name, (p, path) in ssp_paths.items():
    years = path[:, p.i_time]
    T = path[:, p.i_T_AT]
    row = {'scenario': name}
    for th in [1.5, 2.0]:
        above = years[T > th]
        row[f'année_depassement_{th}'] = int(above[0]) if len(above) > 0 else np.nan
    rows.append(row)
df_years = pd.DataFrame(rows)
df_years

## Interprétation

### Éléments de réponse

- Les scénarios SSP se distinguent par leurs hypothèses socio-économiques (croissance de la productivité, démographie, intensité carbone), qui déterminent la trajectoire d'émissions et donc de température.
- Avec seulement 3 trajectoires déterministes, la « probabilité » de dépassement d'un seuil est artificiellement binaire (0 ou 1) : elle ne reflète pas l'incertitude réelle (sensibilité climatique, hypothèses socio-économiques, variabilité naturelle). Une vraie distribution de probabilité nécessite un tirage Monte Carlo sur les paramètres incertains, ce qui est l'objet du TP5.
- La distinction risques chroniques (élévation progressive du niveau moyen de température, montée des eaux) et risques aigus (événements extrêmes) est essentielle en actuariat : les scénarios SSP renseignent surtout sur la trajectoire moyenne (risques chroniques), alors que les risques aigus nécessitent des modèles complémentaires (catastrophe models).